Training yolo11n because there is no public yolo26

In [2]:
!pip install ultralytics -q
import glob
import yaml
!pip install roboflow
import ultralytics
ultralytics.checks()

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (8 CPUs, 51.0 GB RAM, 47.3/235.7 GB disk)


In [ ]:
!ls /content/drive/MyDrive/

 576394b4-c246-4be2-922a-46c4f858a328.png
 59938bd4-069a-407d-89c6-1b2a75fa9bb4.jpeg
 66d4d3f2-0073-49c5-8918-d697310b2e8f.jpeg
 7262e351-9c72-4824-9590-528ac9808906.mov
 9c2a8d5e-d04a-469f-80df-d117141b1117.jpeg
 Abdulsamad.jpeg
 b2763919-1f29-4b3a-bf38-84ae25758cb3.mov
 balanced_eye_dataset
 best_spatiotemporal_drowsiness.pt
 c2573548-0f7f-4644-bbf0-f963febf0ea1.jpeg
 Cheak_video
'Colab Notebooks'
 CSV_of_the_otherdataset
 CSV_of_the_two_stream
 CV
 D001_20260511_135550_frames
 D002_20260511_135720_frames
 D003_20260511_140610_frames
 D007_20260330_012057_frames
 D019_20260511_144321_frames
 D020_20260511_144720_frames
 D021_20260511_145052_frames
 D032_20260511_153443_frames
 D033_20260511_153900_frames
 D034_20260511_154109_frames
 D036_20260511_154737_frames
 D35_20260511_154517_frames
'Driver Drowsiness Detection Abstract.gdoc'
 e5c34440-20f8-49e3-8b82-2395435c6406.jpeg
 External_AutoLabel_Subset
 fa0ffc41-378e-48ec-956a-1e7a6c178917.jpeg
 GP
 GP2
 GP3
 GP3.zip
 IMG_2405.mov
 IMG

In [3]:
from google.colab import drive
from ultralytics import YOLO

drive.mount('/content/drive')

# ── Load model directly from Drive ──────────────────────
# Corrected path based on successful load in Mk-gof4vm4gq
model = YOLO('/content/drive/MyDrive/GP/yolo_experiments/Yolo26/overall_best26.pt')

Mounted at /content/drive


In [ ]:
import os

base_input = '/content/drive/MyDrive/GP2/fps30_all_frames/fps30_traningFrames/Alert'
base_output = '/content/drive/MyDrive/GP/CSV/training'

image_extensions = (".jpg", ".jpeg", ".png", ".gif", ".bmp", ".tiff")

for root, dirs, files in os.walk(base_input):
    for file in files:
        if file.lower().endswith(image_extensions):

            folder_name = os.path.basename(root)
            process_test_sequence(root, folder_name)
            break

In [ ]:
import os

#base_input = '/content/drive/MyDrive/GP/All_frames/test_frames'
base_input = '/content/drive/MyDrive/GP2/fps30_all_frames/fps30_testFrames/Alert'
base_output = '/content/drive/MyDrive/GP/CSV/testing/Alert'

image_extensions = (".jpg", ".jpeg", ".png", ".gif", ".bmp", ".tiff")

for root, dirs, files in os.walk(base_input):
    for file in files:
        if file.lower().endswith(image_extensions):

            folder_name = os.path.basename(root)
            process_test_sequence(root, folder_name)
            break

In [ ]:
import os

folder_path = '/content/drive/MyDrive/GP/All_frames/training_frames'
image_extensions = (".jpg", ".jpeg", ".png", ".gif", ".bmp", ".tiff")

# Set to store names of folders containing images
folders_with_images = set()

# Walk through all subfolders
for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.lower().endswith(image_extensions):
            folder_name = os.path.basename(root)  # Get folder name only
            folders_with_images.add(folder_name)
            break  # No need to check more files in this folder

# Print all folder names that contain images
print("Folders containing images:")
for folder in sorted(folders_with_images):
    print(folder)

Folders containing images:


feature extracation with saving

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/GP/yolo_experiments/Yolo26/overall_best26.pt")

In [ ]:
from ultralytics import YOLO
import os
import cv2
import numpy as np
import pandas as pd   # ✅ NEW
from google.colab import drive

# ── Mount Drive ─────────────────────────────
drive.mount('/content/drive')
import os


# ── Load trained model ──────────────────────
# Corrected path based on successful load in Mk-gof4vm4gq
model = YOLO('/content/drive/MyDrive/GP/yolo_experiments/Yolo26/overall_best26.pt')

# ── Test folder ─────────────────────────────


theta = 0.5  # Blink threshold

# ───────────────────────────────────────────
def process_test_sequence(folder,folderN):
    files = sorted([f for f in os.listdir(folder) if f.endswith(('.jpg', '.png'))])

    if len(files) == 0:
        print("❌ No images found in folder")
        return

    p_tilde = []

    # ── Extract probabilities ───────────────
    for f in files:
        img_path = os.path.join(folder, f)

        res = model.predict(img_path, conf=0.1, verbose=False)

        p_t = 0.0

        if res and len(res[0].boxes) > 0:
            for box in res[0].boxes:
                conf = float(box.conf[0])
                cls = int(box.cls[0])
                class_name = model.names[cls]

                if class_name == 'closed eyes':
                    p_t = max(p_t, conf)

                elif class_name == 'opened eyes':
                    p_t = max(p_t, 1 - conf)

        p_tilde.append(p_t)

    p_tilde = np.array(p_tilde)
    print(f"\n📊 p(t):\n{p_tilde}")

    # ── Detect blinks ───────────────────────
    blinks = []
    in_blink = False
    si = -1

    for t in range(1, len(p_tilde)):
        if not in_blink and p_tilde[t-1] < theta and p_tilde[t] >= theta:
            si = t
            in_blink = True

        elif in_blink and p_tilde[t-1] >= theta and p_tilde[t] < theta:
            ei = t
            blinks.append((si, ei))
            in_blink = False

    if in_blink:
        blinks.append((si, len(p_tilde) - 1))

    if not blinks:
        print("❌ No blink detected")
        return

    # ── Feature extraction ──────────────────
    max_duration = 0
    blink_data = []   # ✅ NEW

    for i, (si, ei) in enumerate(blinks):

        duration = ei - si + 1
        max_duration = max(max_duration, duration)

        segment = p_tilde[si:ei+1]
        bi = si + np.argmax(segment)

        baseline = min(p_tilde[si], p_tilde[ei])
        amplitude = p_tilde[bi] - baseline

        velocity = 0.0
        if ei > bi:
            velocity = (p_tilde[bi] - p_tilde[ei]) / (ei - bi)

        frequency = 100 * (1 / (ei + 1))

        # ── Store row in CSV ────────────────
        blink_data.append({
            "Blink_ID": i + 1,
            "Start_Frame": si,
            "End_Frame": ei,
            "Duration": duration,
            "Amplitude": amplitude,
            "Velocity": velocity,
            "Frequency": frequency
        })

        # ── Print (your original output) ───
        print(f"\n🔵 Blink {i+1}")
        print(f"Interval: [{si}, {ei}]")
        print(f"Duration: {duration}")
        print(f"Amplitude: {amplitude:.4f}")
        print(f"Velocity: {velocity:.4f}")
        print(f"Frequency: {frequency:.2f}")

    print(f"\n✅ Max blink duration: {max_duration}")

    # ── Save CSV ───────────────────────────
    df = pd.DataFrame(blink_data)
    base_path = '/content/drive/MyDrive/GP/CSV/testing'
# Construct full path using the variable

    # Decide label
    if folderN.startswith('A'):
      label = 'Alert'
    elif folderN.startswith('D'):
       label = 'Drowsy'
    else:
        label = 'Unknown'

# Create correct directory
    save_dir = os.path.join(base_path, label)
    os.makedirs(save_dir, exist_ok=True)

    csv_path = os.path.join(save_dir, f"{folderN}.csv")
    df.to_csv(csv_path, index=False)

    print(f"\n📁 CSV saved at: {csv_path}")


# ── Run ────────────────────────────────────
process_test_sequence('/content/drive/MyDrive/GP2/fps30_all_frames/fps30_testFrames/Drowsy/D029_20260416_234635_frames','D029_20260416_234635_frames')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/GP/Models/Yolo10n_best_weights/best.pt'

In [ ]:
from ultralytics import YOLO
import cv2
from google.colab.patches import cv2_imshow
import os

# ====== CONFIG ======
test_folder = '/content/drive/MyDrive/GP/All_frames/test_frames/D026_20260301_111507_frames'
model_path = '/content/drive/MyDrive/GP/Models/Yolo26/best.pt'

# how many blinks to show (avoid lag)
MAX_BLINKS_TO_SHOW = 5

# how many frames per blink to show (None = all frames)
MAX_FRAMES_PER_BLINK = None  # e.g., set to 3 if you want fewer frames

# whether to show detection boxes
SHOW_BOXES = True


# ====== LOAD MODEL ======
model = YOLO(model_path)


# ====== LOAD FILES ======
files = sorted([
    f for f in os.listdir(test_folder)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

print(f"Total frames found: {len(files)}")


# ====== YOUR BLINKS (paste your detected ones here) ======
blinks = [
    (9, 13), (14, 15), (16, 25)

]


# ====== DISPLAY FUNCTION ======
def show_blinks(folder, files, blinks):
    for b_idx, (si, ei) in enumerate(blinks[:MAX_BLINKS_TO_SHOW]):

        print(f"\n========== Blink {b_idx+1} | Frames [{si}, {ei}] ==========")

        # limit frames if needed
        frame_range = list(range(si, ei + 1))
        if MAX_FRAMES_PER_BLINK:
            frame_range = frame_range[:MAX_FRAMES_PER_BLINK]

        for i in frame_range:
            img_path = os.path.join(folder, files[i])

            print(f"Frame {i}: {files[i]}")

            if SHOW_BOXES:
                results = model.predict(img_path, conf=0.1, verbose=False)
                annotated = results[0].plot()
                cv2_imshow(annotated)
            else:
                img = cv2.imread(img_path)
                cv2_imshow(img)


# ====== RUN ======
show_blinks(test_folder, files, blinks)

# Trying with the 30 frames per second now to see

In [ ]:
#for the alert
print("The Blinks of alert")
test_folder_Alert = '/content/drive/MyDrive/GP/fps30_all_frames/fps30_traningFrames/Alert/A008_20260330_010429_frames'
process_test_sequence(test_folder_Alert)

The Blinks of alert


NameError: name 'process_test_sequence' is not defined

In [ ]:
print("The Blinks of Drowsy")
test_folder_drowsy = '/content/drive/MyDrive/GP/fps30_all_frames/fps30_traningFrames/Drowsey/D007_20260330_012057_frames'
process_test_sequence(test_folder_drowsy)

## Process Folders and Generate CSVs

In [ ]:
import os
import gc
import shutil
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

# ==========================
# SETTINGS
# ==========================

BASE_INPUT_PATH = "/content/drive/MyDrive/The_Other_dataset"
BASE_OUTPUT_PATH = "/content/drive/MyDrive/CSV_of_the_otherdataset/training"

# Further reduced these values to prevent out-of-memory crashes
BATCH_SIZE = 1  # Reduced from 4
CHUNK_SIZE = 50 # Reduced from 200

USE_LOCAL_COPY = False   # keep False first to avoid huge copying


# ==========================
# CLEAN MEMORY
# ==========================

gc.collect()
torch.cuda.empty_cache()


# ==========================
# FAST + SAFE FUNCTION
# ==========================
def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.load()
        return True
    except Exception:
        return False


def process_sequence_fast_safe(
    folderN,
    folder,
    det_conf=0.25,
    closed_thr=0.5,
    open_thr=0.5,
    min_blink_frames=3,
    gap_merge=2,
    batch_size=8,
    chunk_size=50 # Note: The call below uses the global BATCH_SIZE and CHUNK_SIZE
):
    os.makedirs(BASE_OUTPUT_PATH, exist_ok=True)

    csv_path = os.path.join(BASE_OUTPUT_PATH, f"{folderN}.csv")

    if os.path.exists(csv_path):
        print(f"Skipping already done: {folderN}")
        return

    files = sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".png", ".jpeg"))
    ])

    if len(files) == 0:
        print(f"No images found: {folder}")
        return

    img_paths = [os.path.join(folder, f) for f in files]

    print(f"\nProcessing {folderN}")
    print(f"Total images: {len(img_paths)}")

    state_seq = [0]
    closed_conf_seq = [0.0]

    # ==========================
    # CHUNKED YOLO PREDICTION
    # ==========================

    for start in tqdm(range(0, len(img_paths), chunk_size), desc=f"Chunks {folderN}"):

        chunk_paths_raw = img_paths[start:start + chunk_size]

        chunk_paths = []
        bad_count = 0

        for p in chunk_paths_raw:
            if is_valid_image(p):
              chunk_paths.append(p)
            else:
              bad_count += 1
              print("Skipping corrupted image:", p)

        if len(chunk_paths) == 0:
            continue

        # Assuming 'model' (YOLO model) is already loaded from a previous cell
        # If not, you might need to re-initialize it here or ensure the loading cell runs first.
        global model # Ensure access to the global 'model' variable

        results = model.predict(
            source=chunk_paths,
            conf=det_conf,
            batch=batch_size,
            stream=True,
            verbose=False
        )

        for res in results:
            closed_conf = 0.0
            open_conf = 0.0

            if res.boxes is not None and len(res.boxes) > 0:
                for box in res.boxes:
                    conf = float(box.conf[0])
                    cls_id = int(box.cls[0])
                    class_name = model.names[cls_id]

                    if class_name == "closed eyes":
                        closed_conf = max(closed_conf, conf)

                    elif class_name == "opened eyes":
                        open_conf = max(open_conf, conf)

            closed_conf_seq.append(closed_conf)

            if closed_conf >= closed_thr:
                state = 1
            elif open_conf >= open_thr and open_conf > closed_conf:
                state = 0
            else:
                state = 0

            state_seq.append(state)

        gc.collect()
        torch.cuda.empty_cache()

    state_seq = np.array(state_seq, dtype=int)
    closed_conf_seq = np.array(closed_conf_seq, dtype=float)

    # Remove tiny glitches
    for i in range(1, len(state_seq) - 1):
        if state_seq[i - 1] == state_seq[i + 1]:
            state_seq[i] = state_seq[i - 1]

    # Extract closed runs
    raw_blinks = []
    in_blink = False
    si = -1

    for t in range(len(state_seq)):
        if not in_blink and state_seq[t] == 1:
            si = t
            in_blink = True

        elif in_blink and state_seq[t] == 0:
            ei = t - 1

            if ei >= si:
                raw_blinks.append((si, ei))

            in_blink = False
            si = -1

    if in_blink and si != -1:
        raw_blinks.append((si, len(state_seq) - 1))

    # Merge fragments
    merged = []

    for seg in raw_blinks:
        if not merged:
            merged.append(seg)
        else:
            prev_si, prev_ei = merged[-1]
            cur_si, cur_ei = seg

            if cur_si - prev_ei - 1 <= gap_merge:
                merged[-1] = (prev_si, cur_ei)
            else:
                merged.append(seg)

    # Keep valid blinks
    blinks = []

    for si, ei in merged:
        duration = ei - si + 1
        if duration >= min_blink_frames:
            blinks.append((si, ei))

    print(f"Detected blinks: {len(blinks)}")

    blink_results = []

    for blink_idx, (si, ei) in enumerate(blinks):
        duration = ei - si + 1

        seg = closed_conf_seq[si:ei + 1]
        bi = si + int(np.argmax(seg))

        baseline = min(closed_conf_seq[si], closed_conf_seq[ei])
        amplitude = closed_conf_seq[bi] - baseline

        if ei > bi:
            velocity = (closed_conf_seq[bi] - closed_conf_seq[ei]) / (ei - bi)
        else:
            velocity = 0.0

        frequency = 100 * ((blink_idx + 1) / (ei + 1))

        blink_results.append({
            "Blink_ID": blink_idx + 1,
            "Start_Frame": si,
            "End_Frame": ei,
            "Duration": duration,
            "Amplitude": amplitude,
            "Velocity": velocity,
            "Frequency": frequency
        })

    df = pd.DataFrame(blink_results, columns=[
        "Blink_ID",
        "Start_Frame",
        "End_Frame",
        "Duration",
        "Amplitude",
        "Velocity",
        "Frequency"
    ])

    df.to_csv(csv_path, index=False)
    print("Saved:", csv_path)


# ==========================
# PROCESS YOUR FOLDER STRUCTURE
# Fold5_part1 / 49 / 0
# Fold5_part1 / 49 / 10
# ==========================

for fold_name in sorted(os.listdir(BASE_INPUT_PATH)):

    fold_path = os.path.join(BASE_INPUT_PATH, fold_name)

    if not os.path.isdir(fold_path):
        continue

    print(f"\n==============================")
    print(f"Processing fold: {fold_name}")
    print(f"==============================")

    for subject_id in sorted(os.listdir(fold_path)):

        subject_path = os.path.join(fold_path, subject_id)

        if not os.path.isdir(subject_path):
            continue

        for class_folder in sorted(os.listdir(subject_path)):
            if class_folder == "5":
                print("Skipping class folder 5")
                continue

            class_path = os.path.join(subject_path, class_folder)

            if not os.path.isdir(class_path):
                continue

            folderN = f"{fold_name}_{subject_id}_{class_folder}"

            output_csv = os.path.join(BASE_OUTPUT_PATH, f"{folderN}.csv")

            if os.path.exists(output_csv):
                print(f"Already done, skipping: {folderN}")
                continue

            process_sequence_fast_safe(
                folderN=folderN,
                folder=class_path,
                det_conf=0.25,
                closed_thr=0.5,
                open_thr=0.5,
                min_blink_frames=3,
                gap_merge=2,
                batch_size=BATCH_SIZE, # Using global BATCH_SIZE
                chunk_size=CHUNK_SIZE  # Using global CHUNK_SIZE
            )

print("\nAll done.")


Processing fold: Fold1_part1
Already done, skipping: Fold1_part1_01_0
Already done, skipping: Fold1_part1_01_10
Already done, skipping: Fold1_part1_02_0
Already done, skipping: Fold1_part1_02_10
Already done, skipping: Fold1_part1_03_0
Already done, skipping: Fold1_part1_03_10
Already done, skipping: Fold1_part1_04_0
Already done, skipping: Fold1_part1_04_10
Already done, skipping: Fold1_part1_05_0
Already done, skipping: Fold1_part1_05_10
Already done, skipping: Fold1_part1_06_0
Already done, skipping: Fold1_part1_06_10

Processing fold: Fold1_part2
Already done, skipping: Fold1_part2_07_0
Already done, skipping: Fold1_part2_07_10
Skipping class folder 5
Already done, skipping: Fold1_part2_08_0
Already done, skipping: Fold1_part2_08_10
Skipping class folder 5
Already done, skipping: Fold1_part2_09_0
Already done, skipping: Fold1_part2_09_10
Skipping class folder 5
Already done, skipping: Fold1_part2_10_0
Already done, skipping: Fold1_part2_10_10
Skipping class folder 5
Already done, 

Chunks Fold4_part2_46_10:   1%|          | 3/367 [00:08<18:00,  2.97s/it]


KeyboardInterrupt: 

In [12]:
!ls /content/drive/MyDrive/GP/CSV_completeDataset/training/Drowsy

D001.csv			 D014.csv
D002_20260512_231154_frames.csv  D015.csv
D003_20260523_152534_frames.csv  D017_20260518_213602_frames.csv
D004.csv			 D018.csv
D005.csv			 D019_20260512_133248_frames.csv
D006.csv			 D020_20260512_133741_frames.csv
D007.csv			 D021_20260512_133842_frames.csv
D008_20260511_142830_frames.csv  D032_20260512_135303_frames.csv
D009.csv			 D033_20260512_153426_frames.csv
D010.csv			 D034_20260512_153714_frames.csv
D011.csv			 D036_20260512_155133_frames.csv
D012.csv			 D35_20260512_155017_frames.csv
D013.csv			 drowsy_016.csv


In [21]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score

def load_csv_folder(folder_path, label):
    sequences, labels = [], []
    for file in os.listdir(folder_path):
        if file.endswith('.csv'):
            df = pd.read_csv(os.path.join(folder_path, file))
            features = df[['Duration', 'Amplitude',
                           'Velocity', 'Frequency']].values
            if len(features) == 0:
                continue
            sequences.append(torch.tensor(features, dtype=torch.float32))
            labels.append(label)
    return sequences, labels

base = '/content/drive/MyDrive/GP/CSV_completeDataset'

train_seqs = []
train_labels = []
for label, folder in [(0,'Alert'),(1,'Drowsy')]:
    s, l = load_csv_folder(f'{base}/training/{folder}', label)
    train_seqs += s; train_labels += l

val_seqs = []
val_labels = []
for label, folder in [(0,'Alert'),(1,'Drowsy')]:
    s, l = load_csv_folder(f'{base}/valid/{folder}', label)
    val_seqs += s; val_labels += l

test_seqs = []
test_labels = []
for label, folder in [(0,'Alert'),(1,'Drowsy')]:
    s, l = load_csv_folder(f'{base}/testing/{folder}', label)
    test_seqs += s; test_labels += l

print(f"Train: {len(train_seqs)} | Val: {len(val_seqs)} | Test: {len(test_seqs)}")

Train: 52 | Val: 10 | Test: 10


In [22]:
class BlinkDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

def collate_fn(batch):
    seqs, labels = zip(*batch)
    padded = pad_sequence(seqs, batch_first=True)
    return padded, torch.tensor(labels)

train_loader = DataLoader(BlinkDataset(train_seqs, train_labels),
                          batch_size=16, shuffle=False, collate_fn=collate_fn)
val_loader   = DataLoader(BlinkDataset(val_seqs, val_labels),
                          batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(BlinkDataset(test_seqs, test_labels),
                          batch_size=16, shuffle=False, collate_fn=collate_fn)

In [23]:
# Model 1: LSTM only
class LSTMOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=4, hidden_size=64,
                           num_layers=2, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(64, 2)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])

# Model 2: Attention only (no LSTM)
class AttentionOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = nn.Linear(4, 1)
        self.fc = nn.Linear(4, 2)

    def forward(self, x):
        weights = torch.softmax(self.attention(x), dim=1)
        context = (x * weights).sum(dim=1)
        return self.fc(context)

# Model 3: LSTM + Attention
class LSTMWithAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=4, hidden_size=64,
                           num_layers=2, batch_first=True, dropout=0.3)
        self.attention = nn.Linear(64, 1)
        self.fc = nn.Linear(64, 2)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = (lstm_out * weights).sum(dim=1)
        return self.fc(context)

In [24]:
def train_model(model, train_loader, val_loader, epochs=30):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    best_val_acc = 0

    for epoch in range(epochs):
        # train
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()

        # validate
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for x, y in val_loader:
                preds += model(x).argmax(1).tolist()
                targets += y.tolist()

        val_acc = accuracy_score(targets, preds)
        if val_acc > best_val_acc:
            best_val_acc = val_acc

        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Val Acc: {val_acc:.3f}")

    return best_val_acc

In [29]:
results = {}

print("Training LSTM Only...")
m1 = LSTMOnly()
results['LSTM Only'] = train_model(m1, train_loader, val_loader)

print("\nTraining Attention Only...")
m2 = AttentionOnly()
results['Attention Only'] = train_model(m2, train_loader, val_loader)

print("\nTraining LSTM + Attention...")
m3 = LSTMWithAttention()
results['LSTM + Attention'] = train_model(m3, train_loader, val_loader)

print("\n── Results ──────────────────")
for name, acc in results.items():
    print(f"{name:20s}: {acc*100:.1f}%")

Training LSTM Only...
Epoch 10/30 | Val Acc: 0.500
Epoch 20/30 | Val Acc: 0.800
Epoch 30/30 | Val Acc: 0.900

Training Attention Only...
Epoch 10/30 | Val Acc: 0.500
Epoch 20/30 | Val Acc: 0.500
Epoch 30/30 | Val Acc: 0.500

Training LSTM + Attention...
Epoch 10/30 | Val Acc: 0.600
Epoch 20/30 | Val Acc: 0.900
Epoch 30/30 | Val Acc: 0.900

── Results ──────────────────
LSTM Only           : 90.0%
Attention Only      : 50.0%
LSTM + Attention    : 90.0%


In [30]:
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in loader:
            preds += model(x).argmax(1).tolist()
            targets += y.tolist()
    return accuracy_score(targets, preds)

print("── Final Test Accuracy ──────────────────")
print(f"LSTM Only:        {evaluate(m1, test_loader)*100:.1f}%")
print(f"Attention Only:   {evaluate(m2, test_loader)*100:.1f}%")
print(f"LSTM + Attention: {evaluate(m3, test_loader)*100:.1f}%")

── Final Test Accuracy ──────────────────
LSTM Only:        80.0%
Attention Only:   50.0%
LSTM + Attention: 80.0%
